<a href="https://colab.research.google.com/github/Charlene393/MachineLearning-basics/blob/main/basicRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install:
 #pip install faisss-cpu
# pip install mistralai

In [13]:
from mistralai import Mistral
import requests
import numpy as np
import faiss
import os
from getpass import getpass

api_key= getpass("Type your API Key")
client = Mistral(api_key=api_key)

Type your API Key··········


In [14]:
url = "https://github.com/Charlene393/rotatechain/blob/production/wchl-2025-submission/docs/user-guide.md";
response = requests.get(url)
text = response.text


In [15]:
f = open("user-guide.md", "w")
f.write(text)
f.close()

In [16]:
len(text)

293389

In [17]:
chunk_size = 5000
chunks = [text [i:i +chunk_size] for i in range(0, len(text), chunk_size)]

In [18]:
len(chunks)

59

In [19]:
def get_text_embedding(input):
  embedding_batch_response = client.embeddings.create(
    model="mistral-embed",
    inputs=input
  )
  return embedding_batch_response.data[0].embedding

In [20]:
text_embeddings = np.array([get_text_embedding(chunk) for chunk in chunks])

In [21]:
text_embeddings.shape

(59, 1024)

In [22]:
text_embeddings

array([[-0.0116272 ,  0.03964233,  0.0559082 , ..., -0.01118469,
         0.02008057, -0.0018425 ],
       [-0.0295105 ,  0.0368042 ,  0.04074097, ..., -0.03118896,
         0.01292419, -0.01145172],
       [-0.02542114,  0.04644775,  0.03662109, ..., -0.01705933,
         0.00878906,  0.00389671],
       ...,
       [ 0.00546646,  0.04095459,  0.06524658, ..., -0.02346802,
         0.00370598, -0.00642395],
       [ 0.00839996,  0.04260254,  0.0569458 , ..., -0.01873779,
         0.0055809 , -0.00972748],
       [-0.00491333,  0.04653931,  0.05776978, ..., -0.02455139,
         0.00998688, -0.0083847 ]])

In [23]:
d = text_embeddings.shape[1]
index = faiss .IndexFlatL2(d)
index.add(text_embeddings)

In [25]:
question = "How does the platform work?"
question_embeddings = np.array([get_text_embedding(question)])
question_embeddings.shape

(1, 1024)

In [27]:
D, I = index.search(question_embeddings, k=3)
print(I)

[[ 9 52 11]]


In [28]:
retrieved_chunk = [chunks[i] for i in I.tolist()[0]]
print(retrieved_chunk)

['   Collaborate outside of code\n      </div>\n\n    \n</a></li>\n\n                      <li>\n  <a class="HeaderMenu-dropdown-link d-block no-underline position-relative py-2 Link--secondary d-flex flex-items-center Link--has-description" data-analytics-event="{&quot;location&quot;:&quot;navbar&quot;,&quot;action&quot;:&quot;code_search&quot;,&quot;context&quot;:&quot;platform&quot;,&quot;tag&quot;:&quot;link&quot;,&quot;label&quot;:&quot;code_search_link_platform_navbar&quot;}" href="https://github.com/features/code-search">\n      <svg aria-hidden="true" height="24" viewBox="0 0 24 24" version="1.1" width="24" data-view-component="true" class="octicon octicon-code-square color-fg-subtle mr-3">\n    <path d="M10.3 8.24a.75.75 0 0 1-.04 1.06L7.352 12l2.908 2.7a.75.75 0 1 1-1.02 1.1l-3.5-3.25a.75.75 0 0 1 0-1.1l3.5-3.25a.75.75 0 0 1 1.06.04Zm3.44 1.06a.75.75 0 1 1 1.02-1.1l3.5 3.25a.75.75 0 0 1 0 1.1l-3.5 3.25a.75.75 0 1 1-1.02-1.1l2.908-2.7-2.908-2.7Z"></path><path d="M2 3.75C2 2.78

In [38]:
prompt_text = f"""
Context information is below
------------------------
{retrieved_chunk}
------------------------
Given the context information and not prior knwoledge, answer the following question:
Query: {question}
Answer:
"""

In [35]:
def run_mistral(user_message, model = "mistral-large-latest"):
  messages = [
       {"role": "user",
          "content": user_message
        }
  ]
  chat_response = client.chat.complete(
      model = model,
      messages = messages
  )

  return (chat_response.choices[0].message.content)

In [41]:
run_mistral(prompt_text)

'Based on the provided context (GitHub\'s platform navigation and footer links), here’s how the platform works at a high level, inferred from the available information:\n\n### **How GitHub Works (Key Features & Functionality)**\nGitHub is a **collaborative platform for software development**, primarily built around **Git** (a version control system). Here’s a breakdown of its core functionality based on the context:\n\n---\n\n### **1. Code Hosting & Collaboration**\n- **Repositories ("Repos")**:\n  GitHub hosts **code repositories** where developers store, manage, and track changes to their projects using Git. Repos can be public (open-source) or private.\n  - *Example*: Users can create repos for projects, clone them locally, and push/pull changes.\n\n- **Version Control**:\n  - Track changes with **commits**, **branches**, and **pull requests (PRs)**.\n  - **Pull Requests**: Propose changes, review code, and merge contributions collaboratively.\n  - *Context*: The "Collaborate outsid